## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")


## Step 2 — Define Dataset Paths

In [ ]:
RAW_PATH = Path("../data/raw/dataset.csv")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_PATH:", RAW_PATH)
print("PROCESSED_DIR:", PROCESSED_DIR)


## Step 3 — Load Raw Dataset

In [ ]:
df = pd.read_csv(RAW_PATH)

print(f"Shape: {df.shape}")
display(df.head())
display(df.tail())
print("Column names:")
print(df.columns.tolist())


## Step 4 — Initial Dataset Audit

In [ ]:
print("### 4.1 Shape")
print(df.shape)

print("\n### 4.2 Column names")
print(df.columns.tolist())

print("\n### 4.3 Data types")
display(df.dtypes)

print("\n### 4.4 Missing values")
missing_summary = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (df.isna().mean() * 100).round(2).values,
})
display(missing_summary.sort_values(["missing_count", "column"], ascending=[False, True]).reset_index(drop=True))

print("\n### 4.5 Duplicate rows")
print(df.duplicated().sum())

print("\n### 4.6 Unique values")
for col in ["business_code", "cust_number", "invoice_currency", "document type", "cust_payment_terms", "isOpen"]:
    if col in df.columns:
        print(f"{col}: {df[col].nunique(dropna=True)} unique values")
        print(df[col].dropna().unique()[:10])

print("\n### 4.7 Numerical statistics")
display(df.describe().T)


## Step 5 — Make a Working Copy

In [ ]:
raw_shape = df.shape
clean_df = df.copy()

print("Raw shape:", raw_shape)
print("Working dataframe shape:", clean_df.shape)
print("The raw CSV remains unchanged because we are modifying a copy only.")


## Step 6 — Rename Columns to FlowCred Canonical Names

In [ ]:
rename_mapping = {
    "invoice_id": "invoice_id",
    "cust_number": "customer_id",
    "total_open_amount": "invoice_amount",
    "posting_date": "invoice_date",
    "due_in_date": "due_date",
    "clear_date": "payment_date"
}

clean_df = clean_df.rename(columns=rename_mapping)

print("Original column names:")
print(df.columns.tolist())
print("\nNew column names:")
print(clean_df.columns.tolist())
print("\nImportant FlowCred fields:")
print("invoice_id, customer_id, invoice_amount, invoice_date, due_date, payment_date")


## Step 7 — Inspect Required Fields

In [ ]:
required_columns = [
    "invoice_id",
    "customer_id",
    "invoice_amount",
    "invoice_date",
    "due_date",
    "payment_date"
]

display(clean_df[required_columns].head())

required_summary = pd.DataFrame({
    "Field": required_columns,
    "Data Type": clean_df[required_columns].dtypes.astype(str).values,
    "Missing Count": clean_df[required_columns].isna().sum().values,
    "Missing %": (clean_df[required_columns].isna().mean() * 100).round(2).values,
    "Unique Count": clean_df[required_columns].nunique(dropna=True).values,
})

display(required_summary)


## Step 8 — Clean Invoice ID

In [ ]:
clean_df["invoice_id"] = clean_df["invoice_id"].astype("string")

missing_invoice_ids = clean_df["invoice_id"].isna().sum()
unique_invoice_ids = clean_df["invoice_id"].nunique(dropna=True)
duplicate_invoice_ids = clean_df["invoice_id"].duplicated().sum()

print("Missing invoice IDs:", missing_invoice_ids)
print("Unique invoice IDs:", unique_invoice_ids)
print("Duplicate invoice IDs:", duplicate_invoice_ids)

invoice_id_duplicates = clean_df[clean_df["invoice_id"].duplicated(keep=False)].sort_values("invoice_id")
print("\nExamples of repeated invoice IDs:")
display(invoice_id_duplicates[["invoice_id", "customer_id", "invoice_amount", "invoice_date", "due_date", "payment_date"]].head(10))

print("\nImportant: we are not removing these duplicates yet. We only remove exact duplicate rows later.")


## Step 9 — Clean Customer ID

In [ ]:
clean_df["customer_id"] = clean_df["customer_id"].astype("string")

missing_customer_ids = clean_df["customer_id"].isna().sum()
number_of_unique_customers = clean_df["customer_id"].nunique(dropna=True)
top_customers = clean_df["customer_id"].value_counts(dropna=False).head(10).reset_index()
top_customers.columns = ["customer_id", "invoice_count"]

print("Missing customer IDs:", missing_customer_ids)
print("Number of unique customers:", number_of_unique_customers)
print("\nTop 10 customers by invoice count:")
display(top_customers)

print("We do not invent or backfill missing customer IDs.")


## Step 10 — Clean Invoice Amount

In [ ]:
clean_df["invoice_amount"] = pd.to_numeric(clean_df["invoice_amount"], errors="coerce")

amount_summary = pd.DataFrame([
    ["Missing amount count", clean_df["invoice_amount"].isna().sum()],
    ["Zero amount count", (clean_df["invoice_amount"] == 0).sum()],
    ["Negative amount count", (clean_df["invoice_amount"] < 0).sum()],
    ["Minimum amount", clean_df["invoice_amount"].min()],
    ["Maximum amount", clean_df["invoice_amount"].max()],
    ["Median amount", clean_df["invoice_amount"].median()],
], columns=["Metric", "Value"])

display(amount_summary)

print("We do not silently delete negative or zero amounts; they remain visible for review.")


## Step 11 — Convert Invoice Date

In [ ]:
clean_df["invoice_date"] = pd.to_datetime(clean_df["invoice_date"], errors="coerce")

missing_invoice_dates = clean_df["invoice_date"].isna().sum()
min_invoice_date = clean_df["invoice_date"].min()
max_invoice_date = clean_df["invoice_date"].max()
invalid_invoice_dates = missing_invoice_dates

print("Missing invoice dates:", missing_invoice_dates)
print("Minimum invoice date:", min_invoice_date)
print("Maximum invoice date:", max_invoice_date)
print("Invalid dates created by conversion:", invalid_invoice_dates)


## Step 12 — Convert Due Date

In [ ]:
def convert_numeric_date_to_datetime(series):
    converted = []
    for value in series:
        if pd.isna(value):
            converted.append(pd.NaT)
            continue
        if isinstance(value, (int, float, np.number)):
            numeric_value = float(value)
            if np.isfinite(numeric_value) and abs(numeric_value - round(numeric_value)) < 1e-8:
                as_string = str(int(round(numeric_value)))
                if len(as_string) == 8:
                    converted.append(pd.to_datetime(as_string, format="%Y%m%d"))
                    continue
        converted.append(pd.to_datetime(value, errors="coerce"))
    return pd.Series(converted, index=series.index, name=series.name)

clean_df["due_date"] = convert_numeric_date_to_datetime(clean_df["due_date"])

print("Missing due dates:", clean_df["due_date"].isna().sum())
print("Minimum due date:", clean_df["due_date"].min())
print("Maximum due date:", clean_df["due_date"].max())
print("Sample converted due dates:")
display(clean_df["due_date"].head(10))

print("This safely converts values such as 20200210 into proper datetime64 values.")


## Step 13 — Convert Payment/Clear Date

In [ ]:
clean_df["payment_date"] = pd.to_datetime(clean_df["payment_date"], errors="coerce")

print("Missing payment dates:", clean_df["payment_date"].isna().sum())
print("Minimum payment date:", clean_df["payment_date"].min())
print("Maximum payment date:", clean_df["payment_date"].max())

print("Missing payment dates are preserved; they indicate no recorded clearing/payment event.")


## Step 14 — Check Date Relationships

In [ ]:
checks = {
    "invoice_date > due_date": clean_df["invoice_date"] > clean_df["due_date"],
    "payment_date < invoice_date": clean_df["payment_date"] < clean_df["invoice_date"],
    "payment_date < due_date": clean_df["payment_date"] < clean_df["due_date"],
}

for check_name, mask in checks.items():
    count = int(mask.sum())
    percentage = (count / len(clean_df)) * 100 if len(clean_df) else 0
    print(f"\n{check_name}")
    print(f"Number of affected records: {count}")
    print(f"Percentage: {percentage:.2f}%")
    display(clean_df.loc[mask, ["invoice_id", "invoice_date", "due_date", "payment_date"]].head(10))

print("\nThese checks are documented only; no records are automatically removed.")


## Step 15 — Remove Exact Duplicate Rows

In [ ]:
rows_before = len(clean_df)
duplicate_rows = clean_df.duplicated().sum()
print("Duplicate rows before removal:", duplicate_rows)

clean_df = clean_df.drop_duplicates().copy()
rows_after = len(clean_df)
rows_removed = rows_before - rows_after

print("Rows before:", rows_before)
print("Rows removed:", rows_removed)
print("Rows after:", rows_after)

print("Only exact duplicate rows were removed; duplicate invoice IDs alone were not removed.")


## Step 16 — Create Payment Status

In [ ]:
clean_df["invoice_status"] = np.where(clean_df["payment_date"].notna(), "paid", "open")

paid_count = (clean_df["invoice_status"] == "paid").sum()
open_count = (clean_df["invoice_status"] == "open").sum()

status_summary = pd.DataFrame({
    "status": ["paid", "open"],
    "count": [paid_count, open_count],
    "percentage": [paid_count / len(clean_df) * 100, open_count / len(clean_df) * 100],
})

display(status_summary)

print("We do not use isOpen as the final target without validating it against payment_date.")


## Step 17 — Create Delay Days

In [ ]:
clean_df["delay_days"] = (clean_df["payment_date"] - clean_df["due_date"]).dt.days

print("Minimum delay:", clean_df["delay_days"].min())
print("Maximum delay:", clean_df["delay_days"].max())
print("Mean delay:", clean_df["delay_days"].mean())
print("Median delay:", clean_df["delay_days"].median())
print("Missing delay count:", clean_df["delay_days"].isna().sum())

print("Open invoices keep NaN delay_days; we do not fill them with zero.")


## Step 18 — Create Late-Payment Target

In [ ]:
clean_df["is_late"] = np.where(
    clean_df["payment_date"].notna(),
    (clean_df["delay_days"] > 0).astype(int),
    np.nan,
)

late_count = clean_df["is_late"].dropna().eq(1).sum()
on_time_count = clean_df["is_late"].dropna().eq(0).sum()
open_count = (clean_df["invoice_status"] == "open").sum()
paid_count = (clean_df["invoice_status"] == "paid").sum()
late_percentage_paid = (late_count / paid_count * 100) if paid_count else 0

summary = pd.DataFrame({
    "Metric": ["Late invoices", "On-time/early invoices", "Open invoices", "Late-payment percentage among paid invoices"],
    "Value": [late_count, on_time_count, open_count, f"{late_percentage_paid:.2f}%"],
})

display(summary)

print("A missing payment_date remains missing; open invoices are not labeled as late.")


## Step 19 — Validate Derived Targets

In [ ]:
paid_check = clean_df["payment_date"].notna()

assert (
    clean_df.loc[paid_check, "is_late"]
    == (clean_df.loc[paid_check, "delay_days"] > 0).astype(int)
).all()

open_mask = clean_df["invoice_status"] == "open"
assert clean_df.loc[open_mask, "payment_date"].isna().all()
assert clean_df.loc[open_mask, "delay_days"].isna().all()
assert clean_df.loc[open_mask, "is_late"].isna().all()

print("Validation succeeded: paid records match the target logic and open records remain unlabeled as late.")


## Step 20 — Sort Dataset Chronologically

In [ ]:
clean_df = clean_df.sort_values(["invoice_date", "invoice_id"], ascending=[True, True]).reset_index(drop=True)

print("First records:")
display(clean_df.head(5))
print("\nLast records:")
display(clean_df.tail(5))


## Step 21 — Final Missing-Value Audit

In [ ]:
final_missing_audit = pd.DataFrame({
    "column": clean_df.columns,
    "data_type": clean_df.dtypes.astype(str).values,
    "missing_count": clean_df.isna().sum().values,
    "missing_percentage": (clean_df.isna().mean() * 100).round(2).values,
    "unique_count": clean_df.nunique(dropna=True).values,
}).sort_values("missing_percentage", ascending=False)

display(final_missing_audit[
    ["column", "data_type", "missing_count", "missing_percentage", "unique_count"]
].head(20))

print("We pay special attention to invoice_id, customer_id, invoice_amount, invoice_date, due_date, payment_date, delay_days, and is_late.")


## Step 22 — Final Dataset Quality Summary

In [ ]:
original_rows = len(df)
final_rows = len(clean_df)
rows_removed = original_rows - final_rows
original_columns = len(df.columns)
final_columns = len(clean_df.columns)
unique_customers = clean_df["customer_id"].nunique(dropna=True)
unique_invoices = clean_df["invoice_id"].nunique(dropna=True)
paid_invoices = (clean_df["invoice_status"] == "paid").sum()
open_invoices = (clean_df["invoice_status"] == "open").sum()
late_invoices = clean_df["is_late"].dropna().eq(1).sum()
on_time_early_invoices = clean_df["is_late"].dropna().eq(0).sum()
missing_invoice_ids = clean_df["invoice_id"].isna().sum()
missing_customer_ids = clean_df["customer_id"].isna().sum()
missing_invoice_amounts = clean_df["invoice_amount"].isna().sum()
missing_invoice_dates = clean_df["invoice_date"].isna().sum()
missing_due_dates = clean_df["due_date"].isna().sum()
missing_payment_dates = clean_df["payment_date"].isna().sum()

summary_table = pd.DataFrame([
    ["Original rows", original_rows],
    ["Final rows", final_rows],
    ["Rows removed", rows_removed],
    ["Original columns", original_columns],
    ["Final columns", final_columns],
    ["Unique customers", unique_customers],
    ["Unique invoices", unique_invoices],
    ["Paid invoices", paid_invoices],
    ["Open invoices", open_invoices],
    ["Late invoices", late_invoices],
    ["On-time/early invoices", on_time_early_invoices],
    ["Missing invoice IDs", missing_invoice_ids],
    ["Missing customer IDs", missing_customer_ids],
    ["Missing invoice amounts", missing_invoice_amounts],
    ["Missing invoice dates", missing_invoice_dates],
    ["Missing due dates", missing_due_dates],
    ["Missing payment dates", missing_payment_dates],
], columns=["Metric", "Value"])

display(summary_table)


## Step 23 — Display Final Schema

In [ ]:
clean_df.info()

print("\nFinal selected fields preview:")
display(clean_df[
    [
        "invoice_id",
        "customer_id",
        "invoice_amount",
        "invoice_date",
        "due_date",
        "payment_date",
        "delay_days",
        "is_late",
        "invoice_status",
    ]
].head(10))


## Step 24 — Save Clean Dataset

In [ ]:
OUTPUT_PATH = Path("../data/processed/flowcred_cleaned.csv")

clean_df.to_csv(OUTPUT_PATH, index=False)

print("Successfully saved:")
print("data/processed/flowcred_cleaned.csv")
print("File exists:", OUTPUT_PATH.exists())

assert OUTPUT_PATH.exists(), "The cleaned dataset was not saved."


## Step 25 — Final Notebook Message

In [ ]:
final_message = """========================================
FLOWCRED AI PREPROCESSING COMPLETE
========================================

Raw Dataset:
50,000 rows approximately

Output:
data/processed/flowcred_cleaned.csv

Prepared fields:
invoice_id
customer_id
invoice_amount
invoice_date
due_date
payment_date
delay_days
is_late
invoice_status

Ready for:
1. Leakage-safe feature engineering
2. Late-payment classification
3. Payment-delay regression
4. Customer behaviour clustering

NOT READY FOR:
AR cash-inflow forecasting

Reason:
payment_amount is not available in the raw dataset.
========================================"""
print(final_message)
